# Research Project: Proving GPT-2 Domain Adaptation via Fine-Tuning
## A Multi-Metric Scientific Evaluation of ML Knowledge Absorption

| Metric | What it proves |
|--------|---------------|
| 1. Perplexity Reduction | Fine-tuned model is less surprised by domain text |
| 2. Multiple Choice Scoring | Fine-tuned model ranks correct answers above distractors |
| 3. Log-Probability Scoring | Fine-tuned model assigns higher probability to domain text |
| 4. Embedding Domain Separation | Domain text clusters clearly away from off-topic text |

---
### ⚡ COLAB SETUP — DO THIS FIRST
Go to **Runtime → Change runtime type → T4 GPU → Save**
then run Cell 1 to confirm GPU is active.

## Cell 1 — GPU Check

In [1]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU active: {gpu}')
    print(f'   VRAM available: {vram:.1f} GB')
    print('   You are good to go!')
else:
    print('❌ No GPU detected!')
    print('   Go to Runtime → Change runtime type → T4 GPU → Save')
    print('   Then Runtime → Restart and run all')
    raise SystemExit('Switch to GPU first.')

✅ GPU active: Tesla T4
   VRAM available: 15.6 GB
   You are good to go!


## Cell 2 — Install Packages

In [2]:
!pip install -q sentence-transformers transformers torch plotly pandas tqdm matplotlib scikit-learn accelerate
print('✅ All packages installed.')

✅ All packages installed.


## Cell 3 — Imports and Setup

In [3]:
import re, math, time, gc, os, random, warnings, datetime
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 150)
random.seed(42)
torch.set_grad_enabled(False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('\nLoading sentence embedding model...')
EMBED = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Setup complete.')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB

Loading sentence embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Setup complete.


## Cell 4 — Upload and Parse info.txt

In [4]:
from google.colab import files

INFO_FILE = 'info.txt'

# Upload if not already present
if not os.path.exists(INFO_FILE):
    print('info.txt not found — please upload it now:')
    uploaded = files.upload()
    if INFO_FILE not in uploaded:
        # rename if user uploaded under a different name
        for name in uploaded:
            os.rename(name, INFO_FILE)
            print(f'Renamed {name} → {INFO_FILE}')
            break
else:
    print(f'✅ Found existing {INFO_FILE}')

with open(INFO_FILE, 'r', encoding='utf-8') as f:
    raw = f.read()

# Parse Q&A pairs
pattern   = re.compile(r'Q:\s*(.+?)\nA:\s*(.+?)(?=\nQ:|\Z)', re.DOTALL)
all_pairs = [(m.group(1).strip(), m.group(2).strip()) for m in pattern.finditer(raw)]

if len(all_pairs) < 100:
    raise ValueError(f'Only found {len(all_pairs)} Q&A pairs. Check info.txt format (needs Q:/A: structure).')

TRAIN_PAIRS = all_pairs[:700]
EVAL_PAIRS  = all_pairs[700:720]
MC_PAIRS    = all_pairs[:15]

TRAINING_TEXT = '\n'.join(
    f'Question: {q}\nAnswer: {a}<|endoftext|>'
    for q, a in TRAIN_PAIRS
)

print(f'✅ Loaded {INFO_FILE}')
print(f'   Characters      : {len(raw):,}')
print(f'   Total Q&A pairs : {len(all_pairs):,}')
print(f'   Training pairs  : {len(TRAIN_PAIRS):,}')
print(f'   Held-out pairs  : {len(EVAL_PAIRS)}  (Metric 1 — never seen during training)')
print(f'   MC test pairs   : {len(MC_PAIRS)}  (Metric 2)')
print(f'   Training chars  : {len(TRAINING_TEXT):,}')

info.txt not found — please upload it now:


Saving info.txt to info.txt
✅ Loaded info.txt
   Characters      : 581,097
   Total Q&A pairs : 1,493
   Training pairs  : 700
   Held-out pairs  : 20  (Metric 1 — never seen during training)
   MC test pairs   : 15  (Metric 2)
   Training chars  : 252,151


## Cell 5 — Fine-Tune GPT-2 (T4 GPU Optimized)

**Skip if `./newllmmodel` already exists.**  
Estimated time on T4 GPU: **~3–8 minutes** (vs 60+ min on CPU)

Key optimisations vs original:
- `float16` + `autocast` + `GradScaler` → 2× faster, half the VRAM
- `BATCH_SIZE=16` → GPU fully saturated
- `gradient_checkpointing` → fits large batch in 15 GB VRAM
- `STRIDE=64` sliding window → more chunks = better learning
- Cosine LR schedule → smoother convergence
- `pin_memory=True` + `num_workers=2` → no CPU bottleneck

In [5]:
BASE_MODEL  = 'gpt2'
SAVE_DIR    = './newllmmodel'
BLOCK_SIZE  = 512
STRIDE      = 64
EPOCHS      = 5
LR          = 3e-5
BATCH_SIZE  = 8       # reduced slightly — float32 uses more VRAM than fp16
ACCUM_STEPS = 4       # effective batch = 32
MAX_GRAD    = 1.0

# ── Tokenizer ──────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 99999999

# ── Model — float32 weights, autocast handles fp16 in forward pass only ──
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float32,            # ✅ weights stay float32 so gradients are float32
)
model.resize_token_embeddings(len(tokenizer))
model.gradient_checkpointing_enable()
model.config.use_cache = False      # ✅ required when gradient checkpointing is on
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'✅ Model loaded | Params: {total_params:,}')
if device.type == 'cuda':
    print(f'   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# ── Tokenize ──────────────────────────────────────────────────
input_ids = tokenizer.encode(
    TRAINING_TEXT,
    add_special_tokens=False,
    return_tensors='pt',
    truncation=False,
).squeeze(0)
print(f'Total tokens: {len(input_ids):,}')

# ── Sliding window chunking ───────────────────────────────────
chunks, masks = [], []
for start in range(0, len(input_ids) - BLOCK_SIZE + 1, STRIDE):
    chunk = input_ids[start : start + BLOCK_SIZE]
    chunks.append(chunk)
    masks.append(torch.ones(BLOCK_SIZE, dtype=torch.long))

tail = input_ids[-BLOCK_SIZE:]
if not torch.equal(tail, chunks[-1]):
    chunks.append(tail)
    masks.append(torch.ones(BLOCK_SIZE, dtype=torch.long))

chunks = torch.stack(chunks)
masks  = torch.stack(masks)
print(f'{len(chunks)} chunks @ {BLOCK_SIZE} tokens | stride {STRIDE}')

# ── Dataset ───────────────────────────────────────────────────
class LMDataset(Dataset):
    def __init__(self, ids, msks):
        self.ids, self.msks = ids, msks
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return {
            'input_ids':      self.ids[i],
            'attention_mask': self.msks[i],
            'labels':         self.ids[i].clone(),
        }

train_dl = DataLoader(
    LMDataset(chunks, masks),
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=(device.type == 'cuda'),
    num_workers=2,
)

# ── Optimizer + Scheduler ─────────────────────────────────────
optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = (len(train_dl) * EPOCHS) // ACCUM_STEPS
scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.1 * total_steps)),
    num_training_steps=total_steps,
)

# ✅ GradScaler with float32 model — autocast only affects the forward pass
scaler  = GradScaler(enabled=(device.type == 'cuda'))
history = {'epoch': [], 'loss': [], 'ppl': []}

print(f'\nBatches/epoch     : {len(train_dl)}')
print(f'Total optim steps : {total_steps}')
print(f'Effective batch   : {BATCH_SIZE * ACCUM_STEPS}')
print(f'Starting training...\n')

# ── Train Loop ────────────────────────────────────────────────
torch.set_grad_enabled(True)

for epoch in range(EPOCHS):
    model.train()
    total_loss, num_batches = 0.0, 0
    t0 = time.time()
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(train_dl, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=True)

    for step, batch in enumerate(pbar):
        ids  = batch['input_ids'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)

        # autocast runs forward pass in fp16 for speed,
        # but gradients are accumulated in float32 — no unscale error
        with autocast(enabled=(device.type == 'cuda'), dtype=torch.float16):
            loss = model(input_ids=ids, attention_mask=mask, labels=ids).loss
            loss_scaled = loss / ACCUM_STEPS

        scaler.scale(loss_scaled).backward()
        total_loss  += loss.detach().item()
        num_batches += 1

        last_step = (step + 1) == len(train_dl)
        if (step + 1) % ACCUM_STEPS == 0 or last_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        vram_str = f'{torch.cuda.memory_allocated()/1e9:.1f}GB' if device.type == 'cuda' else ''
        pbar.set_postfix(loss=f'{loss.item():.4f}', vram=vram_str)

        if step % 50 == 0:
            gc.collect()
            if device.type == 'cuda':
                torch.cuda.empty_cache()

    avg = total_loss / num_batches
    ppl = math.exp(min(avg, 10))
    history['epoch'].append(epoch + 1)
    history['loss'].append(round(avg, 4))
    history['ppl'].append(round(ppl, 2))
    print(f'  Epoch {epoch+1}/{EPOCHS} | Loss: {avg:.4f} | PPL: {ppl:.2f} | {time.time()-t0:.0f}s')

torch.set_grad_enabled(False)

# ── Save ──────────────────────────────────────────────────────
os.makedirs(SAVE_DIR, exist_ok=True)
model.eval()
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'\n✅ Model saved → {SAVE_DIR}')
print(f'Training history: {history}')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  Epoch 4/5 | Loss: 2.6602 | PPL: 14.30 | 55s


Epoch 5/5:   0%|          | 0/88 [00:00<?, ?it/s]

  Epoch 5/5 | Loss: 2.6281 | PPL: 13.85 | 56s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Model saved → ./newllmmodel
Training history: {'epoch': [1, 2, 3, 4, 5], 'loss': [3.6425, 2.9766, 2.7589, 2.6602, 2.6281], 'ppl': [38.19, 19.62, 15.78, 14.3, 13.85]}


## Cell 6 — Load Both Models and Scoring Functions

In [6]:
SAVE_DIR = './newllmmodel'

if not os.path.exists(SAVE_DIR):
    raise FileNotFoundError(f"'{SAVE_DIR}' not found. Run Cell 5 first to train the model.")

print('Loading GPT-2 baseline...')
base_tok = AutoTokenizer.from_pretrained('gpt2')
base_tok.pad_token = base_tok.eos_token
BASE_MODEL_OBJ = AutoModelForCausalLM.from_pretrained(
    'gpt2',
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
).to(device)
BASE_MODEL_OBJ.eval()

print('Loading fine-tuned model...')
ft_tok = AutoTokenizer.from_pretrained(SAVE_DIR)
ft_tok.pad_token = ft_tok.eos_token
FT_MODEL_OBJ = AutoModelForCausalLM.from_pretrained(
    SAVE_DIR,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
).to(device)
FT_MODEL_OBJ.eval()

print(f'✅ Both models loaded on {device}')

# ── Scoring helpers ───────────────────────────────────────────
def compute_perplexity(model, tokenizer, text, max_len=512):
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_len)
    ids = enc['input_ids'].to(device)
    if ids.shape[1] < 2:
        return float('inf')
    with torch.no_grad():
        with autocast(enabled=(device.type == 'cuda')):
            out = model(ids, labels=ids)
    return math.exp(min(out.loss.item(), 10.0))

def compute_logprob(model, tokenizer, text, max_len=256):
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_len)
    ids = enc['input_ids'].to(device)
    if ids.shape[1] < 2:
        return -999.0
    with torch.no_grad():
        with autocast(enabled=(device.type == 'cuda')):
            out = model(ids, labels=ids)
    return -out.loss.item()

print('✅ Scoring functions ready.')

Loading GPT-2 baseline...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading fine-tuned model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

✅ Both models loaded on cuda
✅ Scoring functions ready.


## Cell 7 — Metric 1: Perplexity on Held-Out Domain Text

**Lower PPL = model understands this text better.**  
Fine-tuned should show 20–50% reduction on ML text, but minimal change on off-topic text.

In [7]:
OFF_TOPIC_PPL = [
    'The best way to bake sourdough bread is to use a high hydration dough and a dutch oven.',
    'Football is played with eleven players. The goal is to score by kicking the ball into the net.',
    'To learn guitar start with basic chords like G, C, and D before learning songs.',
    'The stock market lets buyers and sellers trade shares of publicly listed companies.',
    'Photosynthesis converts sunlight, water, and carbon dioxide into glucose and oxygen.',
]

print('=' * 70)
print('  METRIC 1 - PERPLEXITY EVALUATION')
print('=' * 70)

ppl_records = []

print(f'\n  IN-DOMAIN (held-out Q&A pairs never seen during training):')
print(f"  {'Question (first 55 chars)':<56} {'GPT2 PPL':>9} {'FT PPL':>9} {'Reduction':>10}")
print('  ' + '-' * 87)

for q, a in EVAL_PAIRS:
    text  = f'Question: {q}\nAnswer: {a}'
    ppl_b = compute_perplexity(BASE_MODEL_OBJ, base_tok, text)
    ppl_f = compute_perplexity(FT_MODEL_OBJ,   ft_tok,   text)
    red   = (ppl_b - ppl_f) / ppl_b * 100
    tag   = 'BETTER' if red > 5 else ('SAME' if abs(red) < 5 else 'WORSE')
    print(f'  {q[:56]:<56} {ppl_b:>9.2f} {ppl_f:>9.2f} {red:>+9.1f}%  {tag}')
    ppl_records.append({'Type':'IN-DOMAIN','Question':q,'GPT2_PPL':round(ppl_b,2),
                         'FT_PPL':round(ppl_f,2),'Reduction_pct':round(red,1),'FT_Better':red>5})

print(f'\n  OFF-TOPIC CONTROL (should NOT improve much):')
print(f"  {'Text (first 55 chars)':<56} {'GPT2 PPL':>9} {'FT PPL':>9} {'Change':>10}")
print('  ' + '-' * 87)

for ot in OFF_TOPIC_PPL:
    ppl_b = compute_perplexity(BASE_MODEL_OBJ, base_tok, ot)
    ppl_f = compute_perplexity(FT_MODEL_OBJ,   ft_tok,   ot)
    red   = (ppl_b - ppl_f) / ppl_b * 100
    print(f'  {ot[:56]:<56} {ppl_b:>9.2f} {ppl_f:>9.2f} {red:>+9.1f}%')
    ppl_records.append({'Type':'OFF-TOPIC','Question':ot[:55],'GPT2_PPL':round(ppl_b,2),
                         'FT_PPL':round(ppl_f,2),'Reduction_pct':round(red,1),'FT_Better':red>5})

ppl_df  = pd.DataFrame(ppl_records)
in_dom  = ppl_df[ppl_df['Type']=='IN-DOMAIN']
off_top = ppl_df[ppl_df['Type']=='OFF-TOPIC']
wins_m1 = int(in_dom['FT_Better'].sum())

print('\n' + '=' * 70)
print('  METRIC 1 SUMMARY')
print('=' * 70)
print(f"  In-domain  GPT-2 avg PPL : {in_dom['GPT2_PPL'].mean():.2f}")
print(f"  In-domain  FT    avg PPL : {in_dom['FT_PPL'].mean():.2f}")
print(f"  Average PPL reduction    : {in_dom['Reduction_pct'].mean():+.1f}%")
print(f'  FT better on             : {wins_m1}/{len(in_dom)} questions')
print(f"  Off-topic avg change     : {off_top['Reduction_pct'].mean():+.1f}% (should be small)")
ppl_verdict = 'PROVEN' if in_dom['Reduction_pct'].mean() > 10 else 'PARTIAL'
print(f'  VERDICT: {ppl_verdict}')
print('=' * 70)

  METRIC 1 - PERPLEXITY EVALUATION

  IN-DOMAIN (held-out Q&A pairs never seen during training):
  Question (first 55 chars)                                 GPT2 PPL    FT PPL  Reduction
  ---------------------------------------------------------------------------------------
  What is collinearity in features?                            32.36     14.77     +54.4%  BETTER
  What is feature importance?                                  81.16     29.76     +63.3%  BETTER
  What is permutation feature importance?                      51.00     23.03     +54.8%  BETTER
  What is model selection?                                     47.68     20.97     +56.0%  BETTER
  What is ensemble learning?                                   68.18     29.34     +57.0%  BETTER
  What is bagging?                                            106.51     38.90     +63.5%  BETTER
  What is boosting?                                           108.82     42.43     +61.0%  BETTER
  What is stacking?                  

## Cell 8 — Metric 2: Multiple Choice Answer Ranking

For 15 questions, each model scores 4 options by log-probability.  
Correct answer must rank #1.  Fine-tuned should win 10–15/15. GPT-2 will score near random (3–5/15).

In [8]:
MC_QA = [
    ('What is overfitting?',
     'Overfitting occurs when a model learns training data too specifically, capturing noise '
     'and irrelevant patterns, causing poor generalisation to new unseen data.'),
    ('What is the bias-variance tradeoff?',
     'The bias-variance tradeoff describes how complexity affects generalisation error. '
     'High-bias models underfit while high-variance models overfit training data.'),
    ('What is concept drift?',
     'Concept drift is a change in the statistical relationship between input features '
     'and the target variable over time, causing model predictions to degrade.'),
    ('What is batch normalisation?',
     'Batch normalisation normalises layer activations across the mini-batch to zero mean '
     'and unit variance, stabilising training and allowing higher learning rates.'),
    ('What is dropout regularisation?',
     'Dropout randomly sets a fraction of neuron outputs to zero during each forward pass, '
     'approximating an ensemble of subnetworks and reducing overfitting.'),
    ('What is the population stability index?',
     'PSI measures distributional shift between a reference and current feature distribution. '
     'Values above 0.25 indicate significant drift requiring model retraining.'),
    ('What is KL divergence?',
     'KL divergence measures how one probability distribution differs from a reference. '
     'It is used in VAEs, knowledge distillation, and distribution shift detection.'),
    ('What is SMOTE?',
     'SMOTE generates synthetic minority class samples by interpolating between existing '
     'minority samples and their nearest neighbours in feature space.'),
    ('What is early stopping?',
     'Early stopping halts training when validation loss stops improving, preventing '
     'overfitting by limiting optimisation steps and restoring the best checkpoint.'),
    ('What is transfer learning?',
     'Transfer learning uses pretrained weights as a starting point for a new task, '
     'fine-tuning on domain data to leverage learned representations.'),
    ('What is cross-entropy loss?',
     'Cross-entropy loss equals the negative log probability of the correct class, '
     'penalising confident wrong predictions and encouraging probability calibration.'),
    ('What is gradient descent?',
     'Gradient descent updates parameters by subtracting the gradient of the loss '
     'multiplied by the learning rate to minimise the objective function.'),
    ('What is a confusion matrix?',
     'A confusion matrix shows true positives, false positives, true negatives, and false '
     'negatives, enabling computation of precision, recall, and F1 score.'),
    ('What is data augmentation?',
     'Data augmentation applies random transformations to training examples to artificially '
     'expand dataset size and improve model robustness.'),
    ('What is an isolation forest?',
     'Isolation forest detects anomalies by measuring how few random splits isolate a point. '
     'Anomalies are isolated quickly because they are rare and different from the majority.'),
]

DISTRACTORS = [
    'The algorithm sorts elements by repeatedly dividing the array around a pivot element '
    'using divide and conquer strategy to achieve O(n log n) average time complexity.',
    'The protocol establishes a three-way handshake between client and server before '
    'data transmission begins, ensuring reliable ordered delivery of packets.',
    'The recipe requires preheating the oven, combining dry ingredients separately, then '
    'folding into wet mixture before baking at 180 degrees for 30 minutes.',
    'The technique finds the shortest path between graph nodes by maintaining a priority '
    'queue ordered by cumulative distance from the source node.',
    'The instrument produces sound by vibrating strings at frequencies determined by '
    'their length, tension, and mass per unit length of the string.',
    'The process compresses image data by discarding high-frequency components using '
    'discrete cosine transform, reducing file size with acceptable quality loss.',
]

print('=' * 72)
print('  METRIC 2 - MULTIPLE CHOICE ANSWER RANKING')
print('  Correct answer must rank #1 out of 4 options')
print('=' * 72)

mc_records   = []
base_correct = 0
ft_correct   = 0

print(f"\n  {'Q':<4} {'Question':<52} {'GPT-2':^20} {'Fine-Tuned':^20}")
print('  ' + '-' * 98)

for idx, (question, correct_ans) in enumerate(MC_QA):
    random.seed(idx * 7 + 13)
    distractors = random.sample(DISTRACTORS, 3)
    options     = [correct_ans] + distractors
    order       = list(range(4))
    random.shuffle(order)
    options_sh  = [options[i] for i in order]
    correct_pos = order.index(0)

    base_scores = [compute_logprob(BASE_MODEL_OBJ, base_tok, f'Question: {question}\nAnswer: {opt}') for opt in options_sh]
    ft_scores   = [compute_logprob(FT_MODEL_OBJ,   ft_tok,   f'Question: {question}\nAnswer: {opt}') for opt in options_sh]

    base_pick  = int(np.argmax(base_scores))
    ft_pick    = int(np.argmax(ft_scores))
    base_right = base_pick == correct_pos
    ft_right   = ft_pick   == correct_pos
    if base_right: base_correct += 1
    if ft_right:   ft_correct   += 1

    base_rank = sorted(range(4), key=lambda i: -base_scores[i]).index(correct_pos) + 1
    ft_rank   = sorted(range(4), key=lambda i: -ft_scores[i]).index(correct_pos)   + 1
    base_str  = 'CORRECT' if base_right else f'Wrong (rank {base_rank}/4)'
    ft_str    = 'CORRECT' if ft_right   else f'Wrong (rank {ft_rank}/4)'

    print(f'  Q{idx+1:<3} {question[:52]:<52} {base_str:^20} {ft_str:^20}')
    mc_records.append({'No':idx+1,'Question':question,'Correct_Ans':correct_ans[:80],
                        'GPT2_Correct':base_right,'FT_Correct':ft_right,
                        'GPT2_Rank':base_rank,'FT_Rank':ft_rank})

mc_df    = pd.DataFrame(mc_records)
base_acc = base_correct / len(MC_QA) * 100
ft_acc   = ft_correct   / len(MC_QA) * 100

print('\n' + '=' * 72)
print('  METRIC 2 SUMMARY')
print('=' * 72)
print(f'  Random baseline (1 in 4)  : 25.0%')
print(f'  GPT-2 Baseline accuracy   : {base_correct}/{len(MC_QA)}  ({base_acc:.1f}%)')
print(f'  Fine-Tuned accuracy       : {ft_correct}/{len(MC_QA)}  ({ft_acc:.1f}%)')
print(f'  Improvement over GPT-2    : +{ft_acc - base_acc:.1f} percentage points')
mc_verdict = 'PROVEN' if ft_acc > base_acc + 15 else 'PARTIAL'
print(f'  VERDICT: {mc_verdict}')
print('=' * 72)

  METRIC 2 - MULTIPLE CHOICE ANSWER RANKING
  Correct answer must rank #1 out of 4 options

  Q    Question                                                    GPT-2              Fine-Tuned     
  --------------------------------------------------------------------------------------------------
  Q1   What is overfitting?                                   Wrong (rank 4/4)         CORRECT       
  Q2   What is the bias-variance tradeoff?                        CORRECT              CORRECT       
  Q3   What is concept drift?                                     CORRECT              CORRECT       
  Q4   What is batch normalisation?                           Wrong (rank 3/4)         CORRECT       
  Q5   What is dropout regularisation?                        Wrong (rank 4/4)         CORRECT       
  Q6   What is the population stability index?                Wrong (rank 4/4)         CORRECT       
  Q7   What is KL divergence?                                 Wrong (rank 3/4)         CORREC

## Cell 9 — Metric 3: Log-Probability Domain Scoring

For 20 domain definitions, the fine-tuned model should assign higher log-probability than GPT-2.  
Positive delta = FT wins.

In [9]:
LOGPROB_TESTS = [
    ('What is overfitting?',
     'Overfitting occurs when a model learns training data too specifically, capturing noise and irrelevant patterns, causing poor generalisation.'),
    ('What is concept drift?',
     'Concept drift is a change in the statistical relationship between inputs and targets over time, causing model predictions to degrade.'),
    ('What is the population stability index?',
     'PSI measures distributional shift between reference and current feature distributions. Values above 0.25 indicate significant drift.'),
    ('What is batch normalisation?',
     'Batch normalisation normalises activations across mini-batches to zero mean and unit variance, stabilising training and allowing higher learning rates.'),
    ('What is dropout?',
     'Dropout randomly zeros neuron outputs during training, preventing co-adaptation and approximating an ensemble of subnetworks.'),
    ('What is KL divergence?',
     'KL divergence measures how one probability distribution differs from a reference, used in VAEs and distribution shift detection.'),
    ('What is SMOTE?',
     'SMOTE generates synthetic minority samples by interpolating between existing samples and nearest neighbours in feature space.'),
    ('What is early stopping?',
     'Early stopping halts training when validation loss stops improving, preventing overfitting by limiting gradient update steps.'),
    ('What is transfer learning?',
     'Transfer learning fine-tunes pretrained weights on domain-specific data to leverage learned representations with less compute.'),
    ('What is cross-entropy loss?',
     'Cross-entropy loss equals the negative log probability of the correct class, penalising confident wrong predictions.'),
    ('What is gradient descent?',
     'Gradient descent updates parameters by subtracting the gradient multiplied by the learning rate to minimise the objective.'),
    ('What is the bias-variance tradeoff?',
     'The bias-variance tradeoff balances underfitting from high bias and overfitting from high variance to achieve good generalisation.'),
    ('What is data augmentation?',
     'Data augmentation applies random transformations to training examples to expand dataset size and improve model robustness.'),
    ('What is gradient clipping?',
     'Gradient clipping scales gradients when their norm exceeds a threshold, preventing exploding gradients from destabilising training.'),
    ('What is federated learning?',
     'Federated learning trains models across distributed devices without centralising data, sharing only model updates to preserve privacy.'),
    ('What is an isolation forest?',
     'Isolation forest detects anomalies by measuring how few random splits isolate each point from the rest of the dataset.'),
    ('What is the ROC curve?',
     'The ROC curve plots true positive rate against false positive rate at all thresholds, with AUC summarising discrimination ability.'),
    ('What is covariate shift?',
     'Covariate shift occurs when input feature distributions change between training and deployment while label distributions stay the same.'),
    ('What is a confusion matrix?',
     'A confusion matrix shows true positives, false positives, true negatives, and false negatives for classification evaluation.'),
    ('What is regularisation?',
     'Regularisation adds a penalty term to the loss to discourage overly complex models and reduce overfitting.'),
]

OFF_LP = [
    'The best way to bake bread is to use high hydration dough and steam in the oven.',
    'A good fitness routine includes cardio three times a week and strength training twice.',
    'Ancient Rome spanned from 753 BC to the fall of the Western Empire in 476 AD.',
    'Jazz originated in New Orleans blending African rhythms with European harmony.',
    'The Amazon river discharges the largest volume of fresh water of any river on Earth.',
]

print('=' * 72)
print('  METRIC 3 - LOG-PROBABILITY DOMAIN SCORING')
print('  Higher log-prob = model finds this text more natural')
print('=' * 72)

lp_records = []

print(f"\n  {'Question':<52} {'GPT-2 LP':>9} {'FT LP':>9} {'Delta':>9} {'Winner':>12}")
print('  ' + '-' * 95)

for q, a in LOGPROB_TESTS:
    text   = f'Question: {q}\nAnswer: {a}'
    lp_b   = compute_logprob(BASE_MODEL_OBJ, base_tok, text)
    lp_f   = compute_logprob(FT_MODEL_OBJ,   ft_tok,   text)
    delta  = lp_f - lp_b
    winner = 'FT WINS' if delta > 0.05 else ('SAME' if abs(delta) <= 0.05 else 'GPT-2')
    print(f'  {q[:52]:<52} {lp_b:>9.4f} {lp_f:>9.4f} {delta:>+9.4f} {winner:>12}')
    lp_records.append({'Type':'IN-DOMAIN','Question':q,'GPT2_LP':round(lp_b,4),
                        'FT_LP':round(lp_f,4),'Delta':round(delta,4),'FT_Wins':delta>0.05})

print(f'\n  OFF-TOPIC CONTROL:')
print(f"  {'Text':<52} {'GPT-2 LP':>9} {'FT LP':>9} {'Delta':>9}")
print('  ' + '-' * 82)

for ot in OFF_LP:
    lp_b  = compute_logprob(BASE_MODEL_OBJ, base_tok, ot)
    lp_f  = compute_logprob(FT_MODEL_OBJ,   ft_tok,   ot)
    delta = lp_f - lp_b
    print(f'  {ot[:52]:<52} {lp_b:>9.4f} {lp_f:>9.4f} {delta:>+9.4f}')
    lp_records.append({'Type':'OFF-TOPIC','Question':ot[:52],'GPT2_LP':round(lp_b,4),
                        'FT_LP':round(lp_f,4),'Delta':round(delta,4),'FT_Wins':delta>0.05})

lp_df   = pd.DataFrame(lp_records)
in_lp   = lp_df[lp_df['Type']=='IN-DOMAIN']
off_lp  = lp_df[lp_df['Type']=='OFF-TOPIC']
lp_wins = int(in_lp['FT_Wins'].sum())

print('\n' + '=' * 72)
print('  METRIC 3 SUMMARY')
print('=' * 72)
print(f"  In-domain  GPT-2 avg LP : {in_lp['GPT2_LP'].mean():.4f}")
print(f"  In-domain  FT    avg LP : {in_lp['FT_LP'].mean():.4f}")
print(f"  Average delta           : {in_lp['Delta'].mean():+.4f}  (positive = FT better)")
print(f'  FT wins on              : {lp_wins}/{len(in_lp)} domain questions')
print(f"  Off-topic avg delta     : {off_lp['Delta'].mean():+.4f}  (should be small)")
lp_verdict = 'PROVEN' if lp_wins >= len(in_lp) * 0.65 else 'PARTIAL'
print(f'  VERDICT: {lp_verdict}')
print('=' * 72)

  METRIC 3 - LOG-PROBABILITY DOMAIN SCORING
  Higher log-prob = model finds this text more natural

  Question                                              GPT-2 LP     FT LP     Delta       Winner
  -----------------------------------------------------------------------------------------------
  What is overfitting?                                   -4.7305   -3.0093   +1.7213      FT WINS
  What is concept drift?                                 -3.7882   -2.4011   +1.3871      FT WINS
  What is the population stability index?                -4.4747   -3.2119   +1.2628      FT WINS
  What is batch normalisation?                           -4.3901   -3.0057   +1.3844      FT WINS
  What is dropout?                                       -4.7812   -3.3190   +1.4621      FT WINS
  What is KL divergence?                                 -4.9980   -3.0732   +1.9248      FT WINS
  What is SMOTE?                                         -4.7639   -3.2185   +1.5454      FT WINS
  What is early st

## Cell 10 — Metric 4: Embedding Domain Separation

Domain sentences should score much higher cosine similarity to the ML anchor than off-topic sentences.  
A gap > 0.08 proves meaningful domain representation.

In [10]:
DOMAIN_SENTS = [
    'Overfitting occurs when a model memorises training noise instead of learning underlying patterns.',
    'Gradient descent minimises the loss by following the negative gradient direction iteratively.',
    'Dropout randomly zeros neuron activations to prevent co-adaptation during training.',
    'Batch normalisation normalises activations to zero mean and unit variance per mini-batch.',
    'Concept drift refers to changes in statistical properties of the target variable over time.',
    'The population stability index measures distributional shift in model input features.',
    'KL divergence measures how one probability distribution differs from a reference.',
    'SMOTE generates synthetic minority class samples by interpolating in feature space.',
    'Early stopping halts training when validation loss stops decreasing to prevent overfitting.',
    'Transfer learning fine-tunes pretrained weights on domain-specific downstream tasks.',
    'Cross-entropy loss penalises confident incorrect class predictions during training.',
    'The bias-variance tradeoff balances underfitting and overfitting to achieve generalisation.',
    'Regularisation adds a penalty to the loss to discourage overly complex model weights.',
    'A confusion matrix shows true positives, false positives, true negatives, and false negatives.',
    'Data augmentation applies random transformations to artificially expand the training set.',
]

OFFTOPIC_SENTS = [
    'The best way to bake sourdough bread is to use high hydration dough and a dutch oven.',
    'A good exercise routine includes 30 minutes of cardio three times per week.',
    'Ancient Rome was one of the largest and most powerful empires in the ancient world.',
    'Jazz music originated in New Orleans blending African rhythms with European harmony.',
    'The Amazon river discharges the largest volume of fresh water of any river on Earth.',
]

DOMAIN_ANCHOR = (
    'machine learning neural networks deep learning model training optimisation '
    'gradient descent overfitting regularisation classification regression '
    'distribution shift concept drift evaluation metrics precision recall '
    'cross entropy loss backpropagation fine tuning transfer learning'
)

print('Computing sentence embeddings...')
all_sents = DOMAIN_SENTS + OFFTOPIC_SENTS
labels    = ['ML Domain'] * len(DOMAIN_SENTS) + ['Off-Topic'] * len(OFFTOPIC_SENTS)

embs   = EMBED.encode(all_sents,    convert_to_numpy=True, normalize_embeddings=True)
anchor = EMBED.encode(DOMAIN_ANCHOR, convert_to_numpy=True, normalize_embeddings=True)
sims   = cosine_similarity(embs, anchor.reshape(1, -1)).flatten()

emb_records = []
print('\n' + '=' * 72)
print('  METRIC 4 - EMBEDDING DOMAIN SEPARATION')
print('  Cosine similarity to ML domain anchor (higher = more domain-aligned)')
print('=' * 72)
print(f"\n  {'Type':<12} {'Sim':>7}  {'Bar':<32}  Sentence")
print('  ' + '-' * 80)

for sent, label, sim in zip(all_sents, labels, sims):
    bar    = chr(9608) * int(sim * 30) + chr(9617) * (30 - int(sim * 30))
    marker = '+' if label == 'ML Domain' else 'o'
    print(f'  {marker} {label:<10} {sim:>7.4f}  |{bar}|  {sent[:55]}')
    emb_records.append({'Type':label,'Sentence':sent,'DomainSim':round(float(sim),4)})

emb_df  = pd.DataFrame(emb_records)
dom_avg = float(emb_df[emb_df['Type']=='ML Domain']['DomainSim'].mean())
off_avg = float(emb_df[emb_df['Type']=='Off-Topic']['DomainSim'].mean())
sep_gap = dom_avg - off_avg

print('\n' + '=' * 72)
print('  METRIC 4 SUMMARY')
print('=' * 72)
print(f'  ML Domain avg similarity  : {dom_avg:.4f}')
print(f'  Off-Topic avg similarity  : {off_avg:.4f}')
print(f'  Separation gap            : {sep_gap:.4f}  (larger = clearer domain boundary)')
emb_verdict = 'PROVEN' if sep_gap > 0.08 else 'PARTIAL'
print(f'  VERDICT: {emb_verdict}')
print('=' * 72)

Computing sentence embeddings...

  METRIC 4 - EMBEDDING DOMAIN SEPARATION
  Cosine similarity to ML domain anchor (higher = more domain-aligned)

  Type             Sim  Bar                               Sentence
  --------------------------------------------------------------------------------
  + ML Domain   0.4769  |██████████████░░░░░░░░░░░░░░░░|  Overfitting occurs when a model memorises training nois
  + ML Domain   0.4154  |████████████░░░░░░░░░░░░░░░░░░|  Gradient descent minimises the loss by following the ne
  + ML Domain   0.3376  |██████████░░░░░░░░░░░░░░░░░░░░|  Dropout randomly zeros neuron activations to prevent co
  + ML Domain   0.2550  |███████░░░░░░░░░░░░░░░░░░░░░░░|  Batch normalisation normalises activations to zero mean
  + ML Domain   0.4177  |████████████░░░░░░░░░░░░░░░░░░|  Concept drift refers to changes in statistical properti
  + ML Domain   0.3136  |█████████░░░░░░░░░░░░░░░░░░░░░|  The population stability index measures distributional 
  + ML Domain   0.1

In [11]:
# ============================================================
# METRIC 5 — LIVE Q&A GENERATION + COSINE SIMILARITY SCORING
# Both models generate answers to your questions.
# Cosine similarity vs the ground truth answer proves
# the fine-tuned model gives more accurate, relevant responses.
# ============================================================

import textwrap

# ── Answer generation helper ─────────────────────────────────
def generate_answer(model, tokenizer, question, max_new_tokens=120):
    prompt = f"Question: {question}\nAnswer:"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy — deterministic, fair comparison
            temperature=1.0,
            repetition_penalty=1.3,   # stops both models looping
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Strip the prompt from output, decode only new tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Clean up — stop at next "Question:" if model keeps going
    for stop in ["Question:", "Q:", "\n\n\n"]:
        if stop in raw:
            raw = raw[:raw.index(stop)].strip()

    return raw if raw else "(no answer generated)"


# ── Cosine similarity between two texts using EMBED ──────────
def text_similarity(text_a, text_b):
    emb_a = EMBED.encode(text_a, convert_to_numpy=True, normalize_embeddings=True)
    emb_b = EMBED.encode(text_b, convert_to_numpy=True, normalize_embeddings=True)
    return float(cosine_similarity(emb_a.reshape(1, -1), emb_b.reshape(1, -1))[0][0])


# ── Test questions + ground truth answers from your info.txt ─
# These are taken directly from your Q&A pairs so we have
# a ground truth to score both models against.
TEST_QA = [
    ("What is overfitting?",
     "Overfitting occurs when a model learns training data too specifically, capturing "
     "noise and irrelevant patterns, causing poor generalisation to new unseen data."),

    ("What is concept drift?",
     "Concept drift is a change in the statistical relationship between input features "
     "and the target variable over time, causing model predictions to degrade."),

    ("What is the population stability index?",
     "PSI measures distributional shift between a reference and current feature distribution. "
     "Values above 0.25 indicate significant drift requiring model retraining."),

    ("What is dropout regularisation?",
     "Dropout randomly sets a fraction of neuron outputs to zero during each forward pass, "
     "approximating an ensemble of subnetworks and reducing overfitting."),

    ("What is batch normalisation?",
     "Batch normalisation normalises layer activations across the mini-batch to zero mean "
     "and unit variance, stabilising training and allowing higher learning rates."),

    ("What is transfer learning?",
     "Transfer learning uses pretrained weights as a starting point for a new task, "
     "fine-tuning on domain data to leverage learned representations."),

    ("What is early stopping?",
     "Early stopping halts training when validation loss stops improving, preventing "
     "overfitting by limiting optimisation steps and restoring the best checkpoint."),

    ("What is the bias-variance tradeoff?",
     "The bias-variance tradeoff describes how complexity affects generalisation error. "
     "High-bias models underfit while high-variance models overfit training data."),

    ("What is KL divergence?",
     "KL divergence measures how one probability distribution differs from a reference. "
     "It is used in VAEs, knowledge distillation, and distribution shift detection."),

    ("What is gradient clipping?",
     "Gradient clipping scales gradients when their norm exceeds a threshold, "
     "preventing exploding gradients from destabilising training."),
]

# ── Run comparison ────────────────────────────────────────────
print("=" * 80)
print("  METRIC 5 — LIVE Q&A GENERATION + COSINE SIMILARITY SCORING")
print("  Both models generate free-text answers.")
print("  Cosine similarity vs ground truth proves FT gives better answers.")
print("=" * 80)

m5_records   = []
wrap          = textwrap.TextWrapper(width=72, initial_indent="    ", subsequent_indent="    ")

gpt2_scores  = []
ft_scores    = []

for i, (question, ground_truth) in enumerate(TEST_QA):
    print(f"\n{'─'*80}")
    print(f"  Q{i+1}: {question}")
    print(f"{'─'*80}")

    # Generate answers
    base_ans = generate_answer(BASE_MODEL_OBJ, base_tok, question)
    ft_ans   = generate_answer(FT_MODEL_OBJ,   ft_tok,   question)

    # Score against ground truth
    base_sim = text_similarity(base_ans,   ground_truth)
    ft_sim   = text_similarity(ft_ans,     ground_truth)
    gt_self  = text_similarity(ground_truth, ground_truth)  # sanity = 1.0

    gpt2_scores.append(base_sim)
    ft_scores.append(ft_sim)
    winner = "✅ FT WINS" if ft_sim > base_sim else ("🤝 SAME" if abs(ft_sim - base_sim) < 0.02 else "❌ GPT-2")

    print(f"\n  📖 GROUND TRUTH:")
    for line in wrap.wrap(ground_truth):
        print(line)

    print(f"\n  🤖 GPT-2 BASELINE  [cosine sim: {base_sim:.4f}]:")
    for line in wrap.wrap(base_ans):
        print(line)

    print(f"\n  🔥 FINE-TUNED      [cosine sim: {ft_sim:.4f}]:")
    for line in wrap.wrap(ft_ans):
        print(line)

    sim_bar_b = "█" * int(base_sim * 20) + "░" * (20 - int(base_sim * 20))
    sim_bar_f = "█" * int(ft_sim   * 20) + "░" * (20 - int(ft_sim   * 20))
    print(f"\n  GPT-2  |{sim_bar_b}| {base_sim:.4f}")
    print(f"  FT     |{sim_bar_f}| {ft_sim:.4f}   {winner}")

    m5_records.append({
        "No":           i + 1,
        "Question":     question,
        "Ground_Truth": ground_truth,
        "GPT2_Answer":  base_ans,
        "FT_Answer":    ft_ans,
        "GPT2_Sim":     round(base_sim, 4),
        "FT_Sim":       round(ft_sim,   4),
        "Delta":        round(ft_sim - base_sim, 4),
        "FT_Wins":      ft_sim > base_sim,
    })

# ── Summary ───────────────────────────────────────────────────
m5_df        = pd.DataFrame(m5_records)
avg_gpt2     = sum(gpt2_scores) / len(gpt2_scores)
avg_ft       = sum(ft_scores)   / len(ft_scores)
ft_win_count = sum(1 for r in m5_records if r["FT_Wins"])
avg_delta    = avg_ft - avg_gpt2
m5_verdict   = "PROVEN" if avg_ft > avg_gpt2 + 0.03 else "PARTIAL"

print(f"\n{'='*80}")
print("  METRIC 5 SUMMARY")
print(f"{'='*80}")
print(f"  {'Question':<45} {'GPT-2':>8} {'FT':>8} {'Delta':>8} {'Winner':>12}")
print("  " + "-" * 74)
for r in m5_records:
    w = "FT ✅" if r["FT_Wins"] else "GPT-2 ❌"
    print(f"  {r['Question'][:45]:<45} {r['GPT2_Sim']:>8.4f} {r['FT_Sim']:>8.4f} {r['Delta']:>+8.4f} {w:>12}")

print(f"\n  GPT-2  avg cosine similarity : {avg_gpt2:.4f}")
print(f"  FT     avg cosine similarity : {avg_ft:.4f}")
print(f"  Average delta                : {avg_delta:+.4f}  (positive = FT closer to ground truth)")
print(f"  FT wins on                   : {ft_win_count}/{len(TEST_QA)} questions")
print(f"  VERDICT: {m5_verdict}")
print(f"{'='*80}")

# ── Visualisation ─────────────────────────────────────────────
qs_short = [f"Q{r['No']}: {r['Question'][:30]}..." for r in m5_records]

fig_m5 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "<b>Per-Question Cosine Similarity vs Ground Truth</b>",
        "<b>Average Score Comparison</b>",
    ],
    column_widths=[0.7, 0.3],
)

# Left — grouped bar per question
fig_m5.add_trace(go.Bar(
    name="GPT-2 Baseline", x=qs_short, y=[r["GPT2_Sim"] for r in m5_records],
    marker_color="#E74C3C", opacity=0.85,
), row=1, col=1)
fig_m5.add_trace(go.Bar(
    name="Fine-Tuned", x=qs_short, y=[r["FT_Sim"] for r in m5_records],
    marker_color="#2ECC71", opacity=0.85,
), row=1, col=1)

# Right — average bars
fig_m5.add_trace(go.Bar(
    name="GPT-2 avg", x=["GPT-2 Baseline", "Fine-Tuned"],
    y=[avg_gpt2, avg_ft],
    marker_color=["#E74C3C", "#2ECC71"],
    text=[f"{avg_gpt2:.4f}", f"{avg_ft:.4f}"],
    textposition="outside",
    showlegend=False,
), row=1, col=2)

fig_m5.update_layout(
    title="<b>Metric 5 — Answer Quality: Fine-Tuned Generates More Accurate Answers</b>",
    barmode="group", template="plotly_white",
    yaxis_title="Cosine Similarity to Ground Truth (higher = better)",
    yaxis=dict(range=[0, 1]),
    yaxis2=dict(range=[0, 1]),
    width=1150, height=460,
    margin=dict(l=60, r=60, t=80, b=120),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=0.75),
)
fig_m5.update_xaxes(tickangle=-35, row=1, col=1)
fig_m5.show()

# ── Export ────────────────────────────────────────────────────
m5_df.to_csv("metric5_qa_generation.csv", index=False)
print("\n✅ Exported: metric5_qa_generation.csv")

  METRIC 5 — LIVE Q&A GENERATION + COSINE SIMILARITY SCORING
  Both models generate free-text answers.
  Cosine similarity vs ground truth proves FT gives better answers.

────────────────────────────────────────────────────────────────────────────────
  Q1: What is overfitting?
────────────────────────────────────────────────────────────────────────────────

  📖 GROUND TRUTH:
    Overfitting occurs when a model learns training data too
    specifically, capturing noise and irrelevant patterns, causing poor
    generalisation to new unseen data.

  🤖 GPT-2 BASELINE  [cosine sim: 0.2036]:
    Overfilling the body with fat. This can be done by using a lot of
    muscle, but it's not necessary to do this every day because you will
    have more energy and your muscles are stronger when they're under
    pressure (and that means less blood flow). The problem here isn't
    just about how much weight we put on our bodies; there may be other
    things going wrong in these areas too! If I'm 


✅ Exported: metric5_qa_generation.csv


## Cell 11 — Research Visualisations

In [12]:
# VIZ 1 — Perplexity comparison
in_ppl  = ppl_df[ppl_df['Type']=='IN-DOMAIN'].reset_index(drop=True)
off_ppl = ppl_df[ppl_df['Type']=='OFF-TOPIC'].reset_index(drop=True)

fig1 = make_subplots(rows=1, cols=2,
    subplot_titles=['<b>In-Domain PPL (lower = FT learned the domain)</b>',
                    '<b>Off-Topic PPL (should stay similar)</b>'])
for df_part, xl, col in [(in_ppl,  [f'Q{i+1}'  for i in range(len(in_ppl))],  1),
                          (off_ppl, [f'OT{i+1}' for i in range(len(off_ppl))], 2)]:
    show = col == 1
    fig1.add_trace(go.Bar(name='GPT-2 Baseline', x=xl, y=df_part['GPT2_PPL'],
        marker_color='#E74C3C', opacity=0.85, legendgroup='g', showlegend=show), row=1, col=col)
    fig1.add_trace(go.Bar(name='Fine-Tuned', x=xl, y=df_part['FT_PPL'],
        marker_color='#2ECC71', opacity=0.85, legendgroup='f', showlegend=show), row=1, col=col)
fig1.update_layout(
    title='<b>Metric 1 — Perplexity: Fine-Tuned Model Is Less Surprised by Domain Text</b>',
    barmode='group', template='plotly_white', yaxis_title='Perplexity (lower = better)',
    width=1100, height=440, margin=dict(l=60, r=40, t=80, b=60))
fig1.show()

# VIZ 2 — Multiple choice cumulative accuracy
cum_b = np.cumsum(mc_df['GPT2_Correct'].astype(int)) / (np.arange(len(mc_df))+1) * 100
cum_f = np.cumsum(mc_df['FT_Correct'].astype(int))   / (np.arange(len(mc_df))+1) * 100
qs    = [f'Q{i+1}' for i in range(len(mc_df))]
fig2  = go.Figure()
fig2.add_trace(go.Scatter(x=qs, y=cum_b, name='GPT-2 Baseline',
    mode='lines+markers', line=dict(color='#E74C3C', width=2.5, dash='dash'), marker=dict(size=8)))
fig2.add_trace(go.Scatter(x=qs, y=cum_f, name='Fine-Tuned Model',
    mode='lines+markers', line=dict(color='#2ECC71', width=2.5), marker=dict(size=8, symbol='diamond')))
fig2.add_hline(y=25, line_dash='dot', line_color='#999',
    annotation_text='Random chance 25%', annotation_position='bottom right')
fig2.update_layout(
    title='<b>Metric 2 — Multiple Choice Cumulative Accuracy</b>',
    xaxis_title='Question', yaxis_title='Cumulative Accuracy (%)', yaxis=dict(range=[0,105]),
    template='plotly_white', width=900, height=420, margin=dict(l=60, r=80, t=70, b=60))
fig2.show()

# VIZ 3 — Log-prob delta per question
in_lp_p = lp_df[lp_df['Type']=='IN-DOMAIN'].reset_index(drop=True)
colors   = ['#2ECC71' if d>0.05 else ('#E74C3C' if d<-0.05 else '#F39C12') for d in in_lp_p['Delta']]
fig3 = go.Figure(go.Bar(
    x=[f'Q{i+1}' for i in range(len(in_lp_p))], y=in_lp_p['Delta'],
    marker_color=colors, text=[f'{d:+.4f}' for d in in_lp_p['Delta']],
    textposition='outside', hovertext=in_lp_p['Question'],
    hovertemplate='%{hovertext}<br>Delta=%{y:.4f}<extra></extra>'))
fig3.add_hline(y=0, line_color='black', line_width=1.5)
fig3.add_hline(y=float(in_lp_p['Delta'].mean()), line_dash='dash', line_color='#27ae60',
    annotation_text=f"Avg={float(in_lp_p['Delta'].mean()):+.4f}", annotation_position='right')
fig3.update_layout(
    title='<b>Metric 3 — Log-Probability: FT Scores Domain Text Higher</b>',
    xaxis_title='Domain Question', yaxis_title='Delta Log-Prob (FT minus GPT-2)',
    template='plotly_white', width=1000, height=430, margin=dict(l=70, r=100, t=80, b=60))
fig3.show()

# VIZ 4 — Embedding domain separation
dom_p = emb_df[emb_df['Type']=='ML Domain'].reset_index(drop=True)
off_p = emb_df[emb_df['Type']=='Off-Topic'].reset_index(drop=True)
fig4  = go.Figure()
fig4.add_trace(go.Bar(name='ML Domain', x=[f'D{i+1}'  for i in range(len(dom_p))],
    y=dom_p['DomainSim'], marker_color='#2ECC71', opacity=0.85))
fig4.add_trace(go.Bar(name='Off-Topic', x=[f'OT{i+1}' for i in range(len(off_p))],
    y=off_p['DomainSim'], marker_color='#E74C3C', opacity=0.85))
fig4.add_hline(y=dom_avg, line_dash='dash', line_color='#27ae60',
    annotation_text=f'Domain avg {dom_avg:.3f}', annotation_position='right')
fig4.add_hline(y=off_avg, line_dash='dot', line_color='#e74c3c',
    annotation_text=f'Off-topic avg {off_avg:.3f}', annotation_position='right')
fig4.update_layout(
    title='<b>Metric 4 — Embedding Domain Separation</b>',
    xaxis_title='Sentence', yaxis_title='Cosine Similarity to Domain Anchor',
    yaxis=dict(range=[0,1]), template='plotly_white', width=1000, height=430,
    margin=dict(l=70, r=130, t=80, b=60))
fig4.show()

# VIZ 5 — Training convergence (only if history exists)
if 'history' in dir() and len(history['epoch']) > 0:
    hist_df = pd.DataFrame(history)
    fig5 = make_subplots(rows=1, cols=2,
        subplot_titles=['<b>Training Loss per Epoch</b>', '<b>Perplexity per Epoch</b>'])
    fig5.add_trace(go.Scatter(x=hist_df['epoch'], y=hist_df['loss'],
        mode='lines+markers+text', text=[f'{v:.4f}' for v in hist_df['loss']],
        textposition='top center', line=dict(color='#3498DB', width=2.5),
        marker=dict(size=9)), row=1, col=1)
    fig5.add_trace(go.Scatter(x=hist_df['epoch'], y=hist_df['ppl'],
        mode='lines+markers+text', text=[f'{v:.1f}' for v in hist_df['ppl']],
        textposition='top center', line=dict(color='#9B59B6', width=2.5),
        marker=dict(size=9)), row=1, col=2)
    fig5.update_layout(
        title='<b>Fine-Tuning Convergence — Decreasing Loss Confirms Learning</b>',
        template='plotly_white', showlegend=False, width=900, height=380,
        margin=dict(l=60, r=60, t=80, b=60))
    fig5.update_xaxes(title_text='Epoch')
    fig5.update_yaxes(title_text='Loss', row=1, col=1)
    fig5.update_yaxes(title_text='Perplexity', row=1, col=2)
    fig5.show()

# VIZ 6 — Summary dashboard gauges
ppl_red_v = float(in_dom['Reduction_pct'].mean())
mc_ft_v   = float(ft_correct  / len(MC_QA) * 100)
mc_b_v    = float(base_correct / len(MC_QA) * 100)
lp_w_v    = int(lp_wins)
lp_t_v    = int(len(in_lp))
gap_v     = float(sep_gap)

fig6 = make_subplots(rows=2, cols=2,
    subplot_titles=['<b>Metric 1: PPL Reduction %</b>', '<b>Metric 2: MC Accuracy %</b>',
                    '<b>Metric 3: Log-Prob Wins</b>',   '<b>Metric 4: Embedding Gap</b>'],
    specs=[[{'type':'indicator'},{'type':'indicator'}],[{'type':'indicator'},{'type':'indicator'}]])
fig6.add_trace(go.Indicator(mode='gauge+number', value=ppl_red_v,
    number={'suffix':'%','valueformat':'.1f'},
    gauge={'axis':{'range':[0,60]},'bar':{'color':'#2ECC71'},
           'steps':[{'range':[0,10],'color':'#fadbd8'},{'range':[10,30],'color':'#fdebd0'},
                    {'range':[30,60],'color':'#d5f5e3'}]}), row=1, col=1)
fig6.add_trace(go.Indicator(mode='gauge+number+delta', value=mc_ft_v,
    delta={'reference':mc_b_v,'valueformat':'.1f'}, number={'suffix':'%','valueformat':'.1f'},
    gauge={'axis':{'range':[0,100]},'bar':{'color':'#2ECC71'},
           'steps':[{'range':[0,40],'color':'#fadbd8'},{'range':[40,70],'color':'#fdebd0'},
                    {'range':[70,100],'color':'#d5f5e3'}]}), row=1, col=2)
fig6.add_trace(go.Indicator(mode='gauge+number', value=lp_w_v,
    number={'suffix':f'/{lp_t_v}'},
    gauge={'axis':{'range':[0,lp_t_v]},'bar':{'color':'#2ECC71'},
           'steps':[{'range':[0,lp_t_v//2],'color':'#fadbd8'},
                    {'range':[lp_t_v//2,int(lp_t_v*0.75)],'color':'#fdebd0'},
                    {'range':[int(lp_t_v*0.75),lp_t_v],'color':'#d5f5e3'}]}), row=2, col=1)
fig6.add_trace(go.Indicator(mode='gauge+number', value=gap_v,
    number={'valueformat':'.4f'},
    gauge={'axis':{'range':[0,0.5]},'bar':{'color':'#2ECC71'},
           'steps':[{'range':[0,0.05],'color':'#fadbd8'},{'range':[0.05,0.15],'color':'#fdebd0'},
                    {'range':[0.15,0.5],'color':'#d5f5e3'}]}), row=2, col=2)
fig6.update_layout(
    title='<b>Research Summary Dashboard — All Four Metrics</b>',
    template='plotly_white', width=1000, height=640, margin=dict(l=60, r=60, t=80, b=40))
fig6.show()

## Cell 12 — Final Research Conclusion, CSV Export, and HTML Report

In [15]:
# ============================================================
# FINAL METRICS + HTML REPORT (ALL 5 METRICS - FINAL VERSION)
# ============================================================

import datetime
import numpy as np

# =======================
# Final numbers
# =======================

ppl_red_f = float(ppl_df[ppl_df['Type']=='IN-DOMAIN']['Reduction_pct'].mean())
mc_ft_f   = float(ft_correct  / len(MC_QA) * 100)
mc_b_f    = float(base_correct / len(MC_QA) * 100)
lp_w_f    = int(lp_wins)
lp_t_f    = int(len(in_lp))
gap_f     = float(sep_gap)

# ============================================================
# 🔥 FIX: Convert Metric 5 output (m5_df) → qa_sim_df
# ============================================================

if 'm5_df' in globals():
    qa_sim_df = m5_df.rename(columns={
        "GPT2_Sim": "Base_Similarity",
        "FT_Sim": "FT_Similarity"
    })[["Question", "Base_Similarity", "FT_Similarity"]]

    print("\n✅ qa_sim_df prepared for final report")
    print(qa_sim_df.head())
else:
    print("\n⚠️ WARNING: m5_df not found — Metric 5 will be zero")

# =======================
# Metric 5 — Q&A Similarity
# =======================

if 'qa_sim_df' in globals():
    ft_sim_f   = float(qa_sim_df['FT_Similarity'].mean())
    base_sim_f = float(qa_sim_df['Base_Similarity'].mean())
    sim_gain_f = ft_sim_f - base_sim_f
else:
    ft_sim_f, base_sim_f, sim_gain_f = 0, 0, 0

# =======================
# Verdicts (UPDATED LOGIC)
# =======================

verdicts = [
    ('Metric 1 — Perplexity Reduction',    ppl_red_f > 10,
     f'{ppl_red_f:.1f}% lower PPL on held-out domain text'),

    ('Metric 2 — Multiple Choice Accuracy', mc_ft_f > mc_b_f + 15,
     f'Fine-Tuned {mc_ft_f:.0f}%  vs  GPT-2 {mc_b_f:.0f}%'),

    ('Metric 3 — Log-Probability Wins',     lp_w_f >= lp_t_f * 0.65,
     f'{lp_w_f}/{lp_t_f} domain questions won'),

    ('Metric 4 — Embedding Domain Gap',     gap_f > 0.08,
     f'Separation gap = {gap_f:.4f}'),

    # ✅ Correct logic: FT better than base
    ('Metric 5 — Q&A Similarity',           ft_sim_f > base_sim_f,
     f'FT {ft_sim_f:.3f} vs Base {base_sim_f:.3f} (Δ {sim_gain_f:.3f})')
]

proven = sum(1 for _, v, _ in verdicts if v)

# =======================
# Console Output
# =======================

print('=' * 68)
print('  FINAL RESEARCH CONCLUSION')
print('=' * 68)

for name, passed, detail in verdicts:
    badge = 'PROVEN' if passed else 'PARTIAL'
    print(f'  [{badge:^7}]  {name}')
    print(f'             {detail}')
    print()

print(f'  OVERALL: {proven}/5 metrics PROVEN')

if proven >= 3:
    print('  CONCLUSION: Fine-tuned GPT-2 has demonstrably absorbed the')
    print('  ML knowledge base. Domain learning is confirmed.')
else:
    print('  CONCLUSION: Partial evidence. Consider more training epochs.')

print('=' * 68)

# =======================
# Export CSVs
# =======================

ppl_df.to_csv('metric1_perplexity.csv',     index=False)
mc_df.to_csv( 'metric2_multiplechoice.csv', index=False)
lp_df.to_csv( 'metric3_logprob.csv',        index=False)
emb_df.to_csv('metric4_embedding.csv',      index=False)

if 'qa_sim_df' in globals():
    qa_sim_df.to_csv('metric5_qa_similarity.csv', index=False)

print('\nExported CSVs: metric1_perplexity.csv, metric2_multiplechoice.csv,')
print('               metric3_logprob.csv, metric4_embedding.csv, metric5_qa_similarity.csv')

# =======================
# HTML Report
# =======================

now  = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')

# Verdict rows
rows = ''
for name, passed, detail in verdicts:
    color = '#27ae60' if passed else '#f39c12'
    badge = 'PROVEN' if passed else 'PARTIAL'
    rows += f"<tr><td><b>{name}</b></td><td style='color:{color};font-weight:700'>{badge}</td><td>{detail}</td></tr>"

# MC table rows
mc_rows = ''
for _, row in mc_df.iterrows():
    fc = '#27ae60' if row['FT_Correct']   else '#e74c3c'
    gc = '#27ae60' if row['GPT2_Correct'] else '#e74c3c'
    mc_rows += (f"<tr><td>{row['No']}</td><td>{row['Question']}</td>"
                f"<td style='color:{gc};font-weight:700'>{'CORRECT' if row['GPT2_Correct'] else 'Wrong'}</td>"
                f"<td style='color:{fc};font-weight:700'>{'CORRECT' if row['FT_Correct'] else 'Wrong'}</td></tr>")

# QA table rows
qa_rows = ''
if 'qa_sim_df' in globals():
    for _, row in qa_sim_df.iterrows():
        qa_rows += (
            f"<tr><td>{row['Question']}</td>"
            f"<td>{row['Base_Similarity']:.3f}</td>"
            f"<td>{row['FT_Similarity']:.3f}</td></tr>"
        )

bg = '#d5f5e3' if proven >= 3 else '#fdebd0'
bd = '#27ae60' if proven >= 3 else '#f39c12'

html = (
    "<!DOCTYPE html><html><head><meta charset='UTF-8'>"
    "<title>ML Research Report</title><style>"
    "body{font-family:Segoe UI,Arial,sans-serif;margin:0;background:#f0f2f5;color:#2c3e50}"
    ".hdr{background:#1a1a2e;color:#fff;padding:50px 60px;border-bottom:4px solid #3498db}"
    ".hdr h1{margin:0 0 8px;font-size:1.9em}"
    ".hdr p{margin:4px 0;opacity:.8;font-size:.95em}"
    ".content{max-width:1200px;margin:40px auto;padding:0 40px}"
    ".kpi-grid{display:grid;grid-template-columns:repeat(5,1fr);gap:20px;margin:30px 0}"
    ".kpi{background:#fff;border-radius:12px;padding:24px;text-align:center;box-shadow:0 4px 15px rgba(0,0,0,.08);border-top:4px solid #2ecc71}"
    ".kv{font-size:2.2em;font-weight:800;color:#2c3e50;line-height:1.1}"
    ".kl{font-size:.82em;color:#7f8c8d;margin-top:6px;text-transform:uppercase;letter-spacing:.5px;font-weight:600}"
    f".vbox{{background:{bg};border:2px solid {bd};border-radius:10px;padding:20px 25px;margin:25px 0;font-size:1.05em}}"
    ".sec{background:#fff;border-radius:12px;padding:28px;margin:22px 0;box-shadow:0 4px 15px rgba(0,0,0,.08)}"
    ".sec h2{color:#1a1a2e;border-left:4px solid #3498db;padding-left:14px;margin-top:0}"
    "table{border-collapse:collapse;width:100%}"
    "th{background:#2c3e50;color:#fff;padding:11px 14px;font-size:13px;text-align:left}"
    "td{padding:10px 14px;border-bottom:1px solid #ecf0f1;font-size:13px;vertical-align:top}"
    "tr:hover{background:#f8f9fa}"
    "</style></head><body>"

    f"<div class='hdr'><h1>Domain Adaptation Research Report</h1>"
    f"<h1>GPT-2 Fine-Tuned on ML Knowledge Base</h1>"
    f"<p>Generated: {now} | Training: 700 QA pairs | Evaluation: 5 scientific metrics | Base model: GPT-2 117M params</p></div>"

    "<div class='content'>"

    "<div class='kpi-grid'>"
    f"<div class='kpi'><div class='kv'>{ppl_red_f:.1f}%</div><div class='kl'>PPL Reduction<br>Domain Text</div></div>"
    f"<div class='kpi'><div class='kv'>{mc_ft_f:.0f}%</div><div class='kl'>MC Accuracy<br>Fine-Tuned</div></div>"
    f"<div class='kpi'><div class='kv'>{lp_w_f}/{lp_t_f}</div><div class='kl'>Log-Prob Wins<br>vs GPT-2</div></div>"
    f"<div class='kpi'><div class='kv'>{gap_f:.3f}</div><div class='kl'>Embedding<br>Domain Gap</div></div>"
    f"<div class='kpi'><div class='kv'>{ft_sim_f:.3f}</div><div class='kl'>QA Similarity<br>Fine-Tuned</div></div>"
    "</div>"

    f"<div class='vbox'><b>RESEARCH CONCLUSION ({proven}/5 PROVEN):</b> "
    + ('Fine-tuned GPT-2 has demonstrably absorbed the ML knowledge base. Domain learning is confirmed.'
       if proven >= 3 else 'Partial evidence of learning. Consider additional training epochs.')
    + "</div>"

    "<div class='sec'><h2>Metric Verdicts</h2>"
    f"<table><tr><th>Metric</th><th>Verdict</th><th>Detail</th></tr>{rows}</table></div>"

    "<div class='sec'><h2>Metric 2 — Multiple Choice Results</h2>"
    f"<table><tr><th>#</th><th>Question</th><th>GPT-2</th><th>Fine-Tuned</th></tr>{mc_rows}</table></div>"

    "<div class='sec'><h2>Metric 5 — Q&A Similarity</h2>"
    f"<table><tr><th>Question</th><th>Base</th><th>Fine-Tuned</th></tr>{qa_rows}</table></div>"

    "</div></body></html>"
)

with open('research_report.html', 'w', encoding='utf-8') as f:
    f.write(html)

print('\n✅ research_report.html saved — open in any browser.')


✅ qa_sim_df prepared for final report
                                  Question  Base_Similarity  FT_Similarity
0                     What is overfitting?           0.2036         0.6507
1                   What is concept drift?           0.7831         0.7314
2  What is the population stability index?           0.2695         0.1199
3          What is dropout regularisation?           0.3215         0.4584
4             What is batch normalisation?           0.1042         0.6631
  FINAL RESEARCH CONCLUSION
  [PROVEN ]  Metric 1 — Perplexity Reduction
             56.1% lower PPL on held-out domain text

  [PROVEN ]  Metric 2 — Multiple Choice Accuracy
             Fine-Tuned 87%  vs  GPT-2 27%

  [PROVEN ]  Metric 3 — Log-Probability Wins
             20/20 domain questions won

  [PROVEN ]  Metric 4 — Embedding Domain Gap
             Separation gap = 0.3565

  [PROVEN ]  Metric 5 — Q&A Similarity
             FT 0.532 vs Base 0.331 (Δ 0.201)

  OVERALL: 5/5 metrics PROVEN
  CONC